In [1]:
%%capture
%uv pip install flash-attn --no-build-isolation -q 
%uv pip install -q transformers==4.51.3 accelerate==1.2.1 \
    qwen_vl_utils peft==0.11.0 bitsandbytes==0.45.5 \
    scipy scikit-learn pandas pillow torchvision tqdm \
    datasets pyarrow sentencepiece

In [2]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"    Memory: {torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB")
!mkdir -p /root/iqa_unified

PyTorch: 2.8.0+cu129
CUDA: True
GPUs: 1
  GPU 0: NVIDIA A100-SXM4-40GB
    Memory: 42.4 GB


In [3]:
%%writefile /root/iqa_unified/config.py
"""
Configuration for the Unified IQA Framework
A100 40GB settings active.

T4 fallback (change these back for T4):
  - bnb_4bit_compute_dtype / torch_dtype: "float16"
  - attn_implementation: "sdpa"
  - num_groups: 2, num_iterations: 1
  - gradient_accumulation_steps: 4
  - mixed_precision: "fp16"
  - temperature: 0.7
"""

from dataclasses import dataclass, field
from typing import Optional, List
from pathlib import Path


@dataclass
class ModelConfig:
    model_repo: str = "ByteDance/Q-Insight"
    subfolder: str = "score_degradation"
    use_4bit: bool = False                  # A100: no quantization needed
    bnb_4bit_quant_type: str = "nf4"
    bnb_4bit_compute_dtype: str = "bfloat16"
    bnb_4bit_use_double_quant: bool = True
    use_lora: bool = True
    lora_r: int = 8
    lora_alpha: int = 16
    lora_dropout: float = 0.05
    lora_target_modules: List[str] = field(default_factory=lambda: [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ])
    max_new_tokens: int = 256
    torch_dtype: str = "bfloat16"
    attn_implementation: str = "flash_attention_2"
    temperature: float = 1.5                # higher = more diverse candidates = real reward variance
    top_p: float = 0.9
    do_sample: bool = True


@dataclass
class GRPOConfig:
    num_groups: int = 8
    num_iterations: int = 2
    num_epochs: int = 3
    gradient_accumulation_steps: int = 2
    learning_rate: float = 1e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    max_grad_norm: float = 1.0
    clip_range: float = 0.2
    per_device_batch_size: int = 1
    micro_batch_size: int = 1
    gradient_checkpointing: bool = True


@dataclass
class RewardConfig:
    quality_score_weight: float = 1.0
    authenticity_score_weight: float = 1.0
    correspondence_score_weight: float = 1.0
    degradation_class_weight: float = 0.25
    severity_class_weight: float = 0.75
    preference_consistency_weight: float = 0.0
    score_decay_scale: float = 2.0          # smooth exponential decay rate
    partial_severity_reward: float = 0.5
    normalize_rewards: bool = True

    @property
    def total_weight(self):
        return (
            self.quality_score_weight
            + self.authenticity_score_weight
            + self.correspondence_score_weight
            + self.degradation_class_weight
            + self.severity_class_weight
        )


@dataclass
class DataConfig:
    data_dir: str = "/root"
    aigciqa_dir: str = "aigciqa2023"
    kadid_dir: str = "kadid10k"
    image_size: int = 512
    max_pixels: int = 512 * 512
    num_workers: int = 4
    pin_memory: bool = True


@dataclass
class AdversarialConfig:
    attack_type: str = "pgd"
    epsilon: float = 8.0 / 255.0
    alpha: float = 2.0 / 255.0
    num_steps: int = 20
    targeted: bool = False
    score_change_threshold: float = 0.5


@dataclass
class EvalConfig:
    eval_batch_size: int = 1
    max_eval_samples: int = -1
    save_predictions: bool = True
    output_dir: str = "/root/results"
    run_adversarial: bool = False
    cross_dataset: bool = True
    train_datasets: List[str] = field(
        default_factory=lambda: ["aigciqa2023", "kadid10k"]
    )
    eval_datasets: List[str] = field(
        default_factory=lambda: ["aigciqa2023", "kadid10k"]
    )


@dataclass
class UnifiedConfig:
    model: ModelConfig = field(default_factory=ModelConfig)
    grpo: GRPOConfig = field(default_factory=GRPOConfig)
    rewards: RewardConfig = field(default_factory=RewardConfig)
    data: DataConfig = field(default_factory=DataConfig)
    adversarial: AdversarialConfig = field(default_factory=AdversarialConfig)
    eval: EvalConfig = field(default_factory=EvalConfig)

    project_name: str = "Unified-IQA"
    seed: int = 42
    mixed_precision: str = "bf16"
    save_steps: int = 500
    eval_steps: int = 500
    checkpoint_dir: str = "/checkpoints"
    resume_from: Optional[str] = None

    def __post_init__(self):
        Path(self.eval.output_dir).mkdir(parents=True, exist_ok=True)
        Path(self.checkpoint_dir).mkdir(parents=True, exist_ok=True)


def get_default_config() -> UnifiedConfig:
    return UnifiedConfig()


def get_debug_config() -> UnifiedConfig:
    cfg = UnifiedConfig()
    cfg.grpo.num_epochs = 1
    cfg.grpo.gradient_accumulation_steps = 2
    cfg.grpo.num_groups = 4
    cfg.grpo.num_iterations = 1
    cfg.eval.max_eval_samples = 50
    cfg.eval.run_adversarial = False
    cfg.model.max_new_tokens = 256
    return cfg

Writing /root/iqa_unified/config.py


In [4]:
%%writefile /root/iqa_unified/prompts.py
from typing import Dict, List, Optional

SYSTEM_PROMPT = (
    "A conversation between User and Assistant. The user asks a question, and the Assistant solves it. "
    "The assistant first thinks about the reasoning process in the mind and then provides the user with the answer. "
    "The reasoning process and answer are enclosed within <think> </think> and <answer> </answer> tags, respectively, "
    "i.e., <think> reasoning process here </think><answer> answer here </answer>."
)

# Used for aigciqa2023 — has quality + authenticity + correspondence GT
# Matches PREFERENCE_PROMPT from Q-Insight's score task
PREFERENCE_PROMPT = (
    "Assess this image from three perspectives: quality, authenticity, and correspondence. "
    "For each perspective, provide a rating between 1 and 5 (rounded to two decimal places), "
    "where 1 = very poor and 5 = excellent. "
    'Return the final answer in JSON format: '
    '{"quality": <score>, "authenticity": <score>, "correspondence": <score>}'
)

# Used for kadid10k — has quality + distortion_class + severity GT
# Matches Q-Insight's combined score+dist task
SCORE_DEGRADATION_PROMPT = (
    "What is your overall rating on the quality of this picture? "
    "The rating should be a float between 1 and 5, rounded to two decimal places, "
    "with 1 representing very poor quality and 5 representing excellent quality. "
    "Also analyze if it contains any of the following distortions: "
    '"noise", "compression", "blur", or "darken". '
    'If a distortion is present, classify its severity as '
    '"slight", "moderate", "obvious", "serious", or "catastrophic". '
    'Return the final answer in JSON format: '
    '{"quality": <score>, "distortion_class": "<type>", "severity": "<level>"}'
)

# Used for evaluation only — asks for everything
UNIFIED_ASSESSMENT_PROMPT = (
    "Perform a comprehensive image quality assessment:\n"
    "1. Rate the overall quality (1-5, float, two decimal places, where 1=very poor, 5=excellent).\n"
    "2. Rate the authenticity — how realistic and naturally-looking the image appears (1-5, float).\n"
    "3. Rate the correspondence — how well the image content matches what is depicted (1-5, float).\n"
    "4. Identify if the image contains any of these distortions: "
    '"noise", "compression", "blur", or "darken". If none, output "null".\n'
    "5. If a distortion is present, classify its severity as: "
    '"slight", "moderate", "obvious", "serious", or "catastrophic". If none, output "null".\n\n'
    "Think carefully about each dimension before scoring.\n\n"
    'Return the final answer in JSON format:\n'
    '{"quality": <float>, "authenticity": <float>, "correspondence": <float>, '
    '"distortion_class": "<string>", "severity": "<string>"}'
)

# Used for quality-only datasets (koniq) — eval only
SCORE_ONLY_PROMPT = (
    "What is your overall rating on the quality of this picture? "
    "The rating should be a float between 1 and 5, rounded to two decimal places, "
    "with 1 representing very poor quality and 5 representing excellent quality. "
    'Return the final answer in JSON format with the following keys: "rating": The score.'
)

# Used for degradation-only eval
DEGRADATION_PROMPT = (
    'Analyze the given image and determine if it contains any of the following distortions: '
    '"noise", "compression", "blur", or "darken". '
    'If a distortion is present, classify its severity as "slight", "moderate", "obvious", "serious", or "catastrophic". '
    'Return the result in JSON format with the following keys: '
    '"distortion_class": The detected distortion (or "null" if none). '
    'and "severity": The severity level (or "null" if none).'
)

# Comparison prompt — not used in training, kept for eval
COMPARISON_PROMPT = (
    "Given Image A and Image B, assess the visual quality of both images, "
    "explain and justify which one is better considering composition and degradation. "
    'Your answer should be "Image A" or "Image B".'
)

# Maps each dataset to its correct training prompt
# aigciqa2023 → PREFERENCE_PROMPT     (quality + auth + corr)
# kadid10k    → SCORE_DEGRADATION_PROMPT (quality + distortion + severity)
PROMPT_FOR_DATASET = {
    "aigciqa2023": PREFERENCE_PROMPT,
    "kadid10k":    SCORE_DEGRADATION_PROMPT,
    "live":        SCORE_DEGRADATION_PROMPT,
    "tid2013":     SCORE_DEGRADATION_PROMPT,
    "koniq":       SCORE_ONLY_PROMPT,
}


def get_prompt_for_dataset(dataset_name: str) -> str:
    return PROMPT_FOR_DATASET.get(dataset_name, UNIFIED_ASSESSMENT_PROMPT)


def build_single_image_message(image, prompt: str) -> List[Dict]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text",  "text": prompt},
        ]},
    ]


def build_comparison_message(image_a, image_b, prompt: str, ref_image=None) -> List[Dict]:
    content = []
    if ref_image is not None:
        content.append({"type": "text",  "text": "Reference Image:"})
        content.append({"type": "image", "image": ref_image})
    content.append({"type": "text",  "text": "Image A:"})
    content.append({"type": "image", "image": image_a})
    content.append({"type": "text",  "text": "Image B:"})
    content.append({"type": "image", "image": image_b})
    content.append({"type": "text",  "text": prompt})
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": content},
    ]

Writing /root/iqa_unified/prompts.py


In [5]:
%%writefile /root/iqa_unified/iqa_datasets.py
import os, csv, random
from prompts import PREFERENCE_PROMPT, SCORE_DEGRADATION_PROMPT
from pathlib import Path
from typing import Optional, Dict, List, Any
from dataclasses import dataclass
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
from tqdm import tqdm


def normalize_mos_to_1_5(mos: float, dataset: str) -> float:
    """Normalize dataset-specific MOS scores to the 1-5 range the model outputs."""
    if dataset == "aigciqa2023":
        # Scores are integers 1-4, normalize to 1-5
        return max(1.0, min(5.0, (mos - 1.0) / 3.0 * 4.0 + 1.0))
    elif dataset == "kadid10k":
        # DMOS already 1-5
        return max(1.0, min(5.0, mos))
    return max(1.0, min(5.0, mos))


# KADID10k: 25 distortion types mapped to our 4 classes
# Verified against the actual dataset (noise col = distortion type 1-25)
KADID_DISTORTION_MAP = {
    1:  "blur",         # Gaussian blur
    2:  "blur",         # Lens blur
    3:  "blur",         # Motion blur
    4:  "darken",       # Color diffusion
    5:  "darken",       # Color shift
    6:  "darken",       # Color quantization
    7:  "darken",       # Color saturation 1
    8:  "darken",       # Color saturation 2
    9:  "compression",  # JPEG2000
    10: "compression",  # JPEG
    11: "noise",        # White noise
    12: "noise",        # White noise in color
    13: "noise",        # Impulse noise
    14: "noise",        # Multiplicative noise
    15: "noise",        # Denoise
    16: "darken",       # Brighten
    17: "darken",       # Darken
    18: "darken",       # Mean shift
    19: "blur",         # Jitter
    20: "noise",        # Non-eccentricity patch
    21: "blur",         # Pixelate
    22: "compression",  # Quantization
    23: "noise",        # Color block
    24: "blur",         # High sharpen
    25: "blur",         # Contract change
}

KADID_SEVERITY_MAP = {
    1: "slight",
    2: "moderate",
    3: "obvious",
    4: "serious",
    5: "catastrophic",
}


@dataclass
class IQASample:
    image_path: str
    image: Optional[Image.Image] = None
    text_prompt: str = ""
    gt_quality: float = 3.0
    gt_authenticity: float = -1.0
    gt_correspondence: float = -1.0
    gt_distortion_class: str = "null"
    gt_severity: str = "null"
    dataset_name: str = "unknown"


class BaseIQADataset(Dataset):
    def __init__(self, data_dir: str, dataset_name: str,
                 max_samples: int = -1, seed: int = 42):
        self.data_dir = Path(data_dir)
        self.dataset_name = dataset_name
        self.samples: List[IQASample] = []
        self.seed = seed

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        sample = self.samples[idx]
        if sample.image is None:
            sample.image = Image.open(sample.image_path).convert("RGB")
        return {
            "image_path":          sample.image_path,
            "image":               sample.image,
            "text_prompt":         sample.text_prompt,
            "gt_quality":          sample.gt_quality,
            "gt_authenticity":     sample.gt_authenticity,
            "gt_correspondence":   sample.gt_correspondence,
            "gt_distortion_class": sample.gt_distortion_class,
            "gt_severity":         sample.gt_severity,
            "dataset_name":        sample.dataset_name,
        }

    def _find_images_dir(self) -> Path:
        for d in ["images", "imgs", "image", "photos"]:
            if (self.data_dir / d).exists(): return self.data_dir / d
        if any(p.suffix.lower() in (".png", ".jpg", ".jpeg", ".bmp", ".webp")
               for p in self.data_dir.iterdir()):
            return self.data_dir
        raise FileNotFoundError(f"Cannot find images directory in {self.data_dir}")


class AIGCIQA2023Dataset(BaseIQADataset):
    """
    AIGCIQA2023+ dataset.

    CSV format (no header row):
      col 0: numeric index (0, 1, 2, ...) — used to construct filename as "{col0}.png"
      col 1: original image name (ignored — filenames on disk are just "{index}.png")
      col 2: model/method name (ignored)
      col 3: quality score      (integer 1-4) → normalized to 1-5
      col 4: authenticity score (integer 1-4) → normalized to 1-5
      col 5: correspondence score (integer 1-4) → normalized to 1-5
    """

    def __init__(self, data_dir: str, max_samples: int = -1, seed: int = 42):
        super().__init__(data_dir, "aigciqa2023", max_samples, seed)

        csv_path = None
        for cand in ["AIGIQA2023+.csv", "AIGCIQA2023+.csv", "annotations.csv", "data.csv"]:
            p = self.data_dir / cand
            if p.exists(): csv_path = p; break
        if csv_path is None:
            raise FileNotFoundError(f"Cannot find CSV in {self.data_dir}")

        images_dir = self._find_images_dir()

        rows = None
        for enc in ["utf-8", "latin-1", "cp1252", "iso-8859-1"]:
            try:
                with open(csv_path, "r", encoding=enc) as f:
                    rows = list(csv.reader(f))
                print(f"  AIGCIQA2023 loaded with encoding={enc}, rows={len(rows)}")
                break
            except UnicodeDecodeError:
                continue
        if rows is None:
            with open(csv_path, "r", encoding="utf-8", errors="ignore") as f:
                rows = list(csv.reader(f))

        if not rows:
            raise ValueError(f"Empty CSV at {csv_path}")

        data_rows = [r for r in rows if len(r) > 5]

        rng = random.Random(seed)
        if max_samples > 0:
            data_rows = rng.sample(data_rows, min(max_samples, len(data_rows)))

        skipped = 0
        for row in tqdm(data_rows, desc="Loading AIGCIQA2023"):
            try:
                file_idx = int(row[0])
            except (ValueError, IndexError):
                skipped += 1
                continue

            fname    = f"{file_idx}.png"
            img_path = images_dir / fname
            if not img_path.exists():
                skipped += 1
                continue

            try:
                raw_quality        = float(row[3])
                raw_authenticity   = float(row[4])
                raw_correspondence = float(row[5])
            except (ValueError, IndexError):
                skipped += 1
                continue

            quality        = normalize_mos_to_1_5(raw_quality,        "aigciqa2023")
            authenticity   = normalize_mos_to_1_5(raw_authenticity,   "aigciqa2023")
            correspondence = normalize_mos_to_1_5(raw_correspondence, "aigciqa2023")

            self.samples.append(IQASample(
                image_path=str(img_path),
                text_prompt=PREFERENCE_PROMPT,
                gt_quality=quality,
                gt_authenticity=authenticity,
                gt_correspondence=correspondence,
                gt_distortion_class="null",
                gt_severity="null",
                dataset_name="aigciqa2023",
            ))

        print(f"  AIGCIQA2023: {len(self.samples)} loaded, {skipped} skipped")


class KADID10kDataset(BaseIQADataset):
    """
    KADID10k dataset.

    CSV columns: image, dmos, reference, noise
      image  → filename e.g. I01_01_03.png
               format: {ref}_{distortion_type:02d}_{severity:02d}.png
      dmos   → quality score, already in 1-5 range
      noise  → distortion type code 1-25 (NOT severity)
               severity is the 3rd part of filename (_01 to _05)

    Verified:
      noise 1-25  → 4 classes (blur/darken/compression/noise)
      filename _01-_05 → slight/moderate/obvious/serious/catastrophic
    """

    def __init__(self, data_dir: str, max_samples: int = -1, seed: int = 42):
        super().__init__(data_dir, "kadid10k", max_samples, seed)

        csv_path = self.data_dir / "image_labeled_by_per_noise.csv"
        if not csv_path.exists():
            csv_path = self.data_dir / "dmos.csv"
        if not csv_path.exists():
            raise FileNotFoundError(...)

        images_dir = self.data_dir / "images"
        if not images_dir.exists():
            raise FileNotFoundError(f"Cannot find images/ directory in {self.data_dir}")

        import pandas as pd
        df = pd.read_csv(csv_path)

        df["distortion_class"] = df["noise"].map(KADID_DISTORTION_MAP)
        df["severity_level"]   = df["image"].apply(
            lambda x: int(x.split("_")[2].split(".")[0]))
        df["severity_label"]   = df["severity_level"].map(KADID_SEVERITY_MAP)

        if max_samples > 0:
            df = df.sample(n=min(max_samples, len(df)), random_state=seed)

        skipped = 0
        for _, row in tqdm(df.iterrows(), total=len(df), desc="Loading KADID10k"):
            img_path = images_dir / row["image"]
            if not img_path.exists():
                skipped += 1
                continue

            try:
                quality = float(row["dmos"])
            except (KeyError, ValueError):
                skipped += 1
                continue

            self.samples.append(IQASample(
                image_path=str(img_path),
                text_prompt=SCORE_DEGRADATION_PROMPT,
                gt_quality=quality,
                gt_authenticity=-1.0,
                gt_correspondence=-1.0,
                gt_distortion_class=row["distortion_class"],
                gt_severity=row["severity_label"],
                dataset_name="kadid10k",
            ))

        print(f"  KADID10k: {len(self.samples)} loaded, {skipped} skipped")


# ------------------------------------------------------------------
# Registry and helpers
# ------------------------------------------------------------------

DATASET_REGISTRY = {
    "aigciqa2023": AIGCIQA2023Dataset,
    "kadid10k":    KADID10kDataset,
}


def create_dataset(name: str, data_dir: str, max_samples: int = -1, seed: int = 42):
    if name not in DATASET_REGISTRY:
        raise ValueError(f"Unknown dataset '{name}'. Available: {list(DATASET_REGISTRY.keys())}")
    return DATASET_REGISTRY[name](data_dir, max_samples, seed)


class CombinedIQADataset(Dataset):
    def __init__(self, datasets: List[BaseIQADataset], seed: int = 42):
        self.datasets = datasets
        self._indices = [(d, s) for d, ds in enumerate(datasets) for s in range(len(ds))]

    def __len__(self): return len(self._indices)

    def __getitem__(self, idx):
        d_idx, s_idx = self._indices[idx]
        return self.datasets[d_idx][s_idx]


def create_combined_dataset(dataset_configs, base_dir="/root",
                             max_samples_per_dataset=-1, seed=42):
    datasets = []
    for cfg in dataset_configs:
        name     = cfg["name"]
        data_dir = cfg.get("dir", os.path.join(base_dir, name))
        max_s    = cfg.get("max_samples", max_samples_per_dataset)
        try:
            ds = create_dataset(name, data_dir, max_samples=max_s, seed=seed)
            if len(ds) > 0:
                datasets.append(ds)
                print(f"  Loaded {name}: {len(ds)} samples")
        except Exception as e:
            print(f"  Warning: {name} error: {e}")
    if not datasets:
        raise RuntimeError("No datasets loaded")
    return CombinedIQADataset(datasets, seed)


def iqa_collate_fn(batch):
    return {
        "images":              [s["image"] for s in batch],
        "text_prompts":        [s["text_prompt"] for s in batch],
        "gt_quality":          torch.tensor([s["gt_quality"] for s in batch],
                                            dtype=torch.float32),
        "gt_authenticity":     torch.tensor([s["gt_authenticity"] for s in batch],
                                            dtype=torch.float32),
        "gt_correspondence":   torch.tensor([s["gt_correspondence"] for s in batch],
                                            dtype=torch.float32),
        "gt_distortion_class": [s["gt_distortion_class"] for s in batch],
        "gt_severity":         [s["gt_severity"] for s in batch],
    }


def create_dataloaders(combined_dataset, batch_size=1, num_workers=4,
                       val_split=0.1, seed=42):
    n       = len(combined_dataset)
    indices = list(range(n))
    random.Random(seed).shuffle(indices)
    val_size = max(1, int(n * val_split))
    train_ds = torch.utils.data.Subset(combined_dataset, indices[val_size:])
    val_ds   = torch.utils.data.Subset(combined_dataset, indices[:val_size])
    return (
        DataLoader(train_ds, batch_size, shuffle=True,  num_workers=num_workers,
                   collate_fn=iqa_collate_fn),
        DataLoader(val_ds,   batch_size, shuffle=False, num_workers=num_workers,
                   collate_fn=iqa_collate_fn),
    )

Writing /root/iqa_unified/iqa_datasets.py


In [6]:
%%writefile /root/iqa_unified/model_utils.py
import torch, json, re, gc, os, ast
from typing import Dict, List, Optional, Any
from PIL import Image
from io import BytesIO
import requests
import warnings
warnings.filterwarnings("ignore")
from transformers import Qwen2_5_VLForConditionalGeneration, Qwen2_5_VLProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from config import ModelConfig
from prompts import build_single_image_message

def load_image(source):
    if isinstance(source, Image.Image): return source.convert("RGB")
    if str(source).startswith("http"):
        r = requests.get(source, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        r.raise_for_status()
        return Image.open(BytesIO(r.content)).convert("RGB")
    return Image.open(source).convert("RGB")

def clear_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    gc.collect()

def print_gpu_memory(prefix=""):
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            print(f"{prefix}GPU {i}: {torch.cuda.memory_allocated(i)/1e9:.2f} GB allocated, "
                  f"{torch.cuda.memory_reserved(i)/1e9:.2f} GB reserved")

def load_model_and_processor(config, device_map="auto"):
    bnb_config = None
    if config.use_4bit:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type=config.bnb_4bit_quant_type,
            bnb_4bit_compute_dtype=getattr(torch, config.bnb_4bit_compute_dtype),
            bnb_4bit_use_double_quant=config.bnb_4bit_use_double_quant,
        )
    torch_dtype = getattr(torch, config.torch_dtype)
    processor = Qwen2_5_VLProcessor.from_pretrained(config.model_repo, subfolder=config.subfolder)
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        config.model_repo, subfolder=config.subfolder,
        quantization_config=bnb_config, device_map=device_map,
        torch_dtype=torch_dtype,
        attn_implementation=config.attn_implementation,
    )
    if config.use_lora:
        if config.use_4bit:
            model = prepare_model_for_kbit_training(model)
        model = get_peft_model(model, LoraConfig(
            r=config.lora_r, lora_alpha=config.lora_alpha, lora_dropout=config.lora_dropout,
            target_modules=config.lora_target_modules, bias="none", task_type=TaskType.CAUSAL_LM,
        ))
        model.print_trainable_parameters()
    model.eval()
    print("Model loaded!")
    print_gpu_memory("  ")
    return model, processor

@torch.no_grad()
def generate_response(model, processor, messages, max_new_tokens=256,
                       temperature=1.0, top_p=0.9, do_sample=True):
    from qwen_vl_utils import process_vision_info
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors="pt")
    first_device = next(model.parameters()).device
    inputs = {k: v.to(first_device) if hasattr(v, "to") else v for k, v in inputs.items()}
    amp_dtype = next(model.parameters()).dtype
    with torch.cuda.amp.autocast(dtype=amp_dtype):
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, use_cache=True,
            do_sample=do_sample,
            temperature=temperature if do_sample else 1.0,
            top_p=top_p if do_sample else 1.0,
            top_k=50 if do_sample else None,
        )
    trimmed = [out[len(inp):] for inp, out in zip(inputs["input_ids"], outputs)]
    return processor.batch_decode(trimmed, skip_special_tokens=True)[0]

def parse_thinking(response: str) -> str:
    # <think> format
    m = re.search(r"<think>(.*?)</think>", response, re.DOTALL)
    if m: return m.group(1).strip()
    # Q-Insight native: REASONING: ... (up to the === divider or ANSWER:)
    m = re.search(r"REASONING:(.*?)(?:={4,}|ANSWER:)", response, re.DOTALL)
    if m: return m.group(1).strip()
    # Fallback: everything before <answer>
    m = re.search(r"(.*?)<answer>", response, re.DOTALL)
    return m.group(1).strip() if m else ""

def parse_answer(response: str) -> Dict[str, Any]:
    # Priority 1: <answer>...</answer> tags (target format)
    m = re.search(r"<answer>(.*?)</answer>", response, re.DOTALL)
    if m:
        return parse_json_from_response(m.group(1).strip())

    # Priority 2: Q-Insight native format  →  ANSWER: {...}
    m = re.search(r"ANSWER:\s*(\{[^}]+\})", response, re.DOTALL)
    if m:
        return parse_json_from_response(m.group(1).strip())

    # Priority 3: try to parse the whole response
    return parse_json_from_response(response)

def parse_json_from_response(text: str) -> Dict[str, Any]:
    # --- Pass 1: strict JSON ---
    try:
        return _normalise(json.loads(text))
    except (json.JSONDecodeError, ValueError):
        pass

    # --- Pass 2: Python dict literal (Q-Insight uses single-quoted dicts) ---
    # ast.literal_eval is safe — only handles literals, no code execution
    try:
        result = ast.literal_eval(text.strip())
        if isinstance(result, dict):
            return _normalise(result)
    except Exception:
        pass

    # --- Pass 3: JSON inside markdown code fences ---
    m = re.search(r"```(?:json)?\s*([\s\S]*?)```", text)
    if m:
        try:
            return _normalise(json.loads(m.group(1).strip()))
        except (json.JSONDecodeError, ValueError):
            pass

    # --- Pass 4: extract first {...} block and try JSON then ast ---
    m = re.search(r"\{[^{}]*\}", text, re.DOTALL)
    if m:
        blob = m.group(0)
        try:
            return _normalise(json.loads(blob))
        except (json.JSONDecodeError, ValueError):
            pass
        try:
            result = ast.literal_eval(blob)
            if isinstance(result, dict):
                return _normalise(result)
        except Exception:
            pass

    # --- Pass 5: regex field-by-field extraction (handles mixed quoting) ---
    # Matches both "key" and 'key' with either quoted or bare numeric values
    result: Dict[str, Any] = {"raw": text}
    for pat, key in [
        (r"""['""](?:quality|rating)['"\"]\s*:\s*([\d.]+)""", "quality"),
        (r"""['""]authenticity['"\"]\s*:\s*([\d.]+)""",       "authenticity"),
        (r"""['""]correspondence['"\"]\s*:\s*([\d.]+)""",     "correspondence"),
    ]:
        m = re.search(pat, text)
        if m:
            try:
                result[key] = float(m.group(1))
            except ValueError:
                pass

    for pat, key in [
        (r"""['""]distortion_class['"\"]\s*:\s*['""]?([a-zA-Z_]+)['""]?""", "distortion_class"),
        (r"""['""]severity['"\"]\s*:\s*['""]?([a-zA-Z_]+)['""]?""",         "severity"),
    ]:
        m = re.search(pat, text)
        if m:
            result[key] = m.group(1).strip()

    # Also try bare ANSWER: rating pattern for Q-Insight legacy outputs
    # e.g.  "→ Quality Score: 2.4 / 5.0"
    if "quality" not in result:
        m = re.search(r"Quality\s+Score:\s*([\d.]+)", text, re.IGNORECASE)
        if m:
            try:
                result["quality"] = float(m.group(1))
            except ValueError:
                pass

    return result

def _normalise(d: Dict[str, Any]) -> Dict[str, Any]:
    """Rename 'rating' / 'score' → 'quality' so the rest of the pipeline
    always sees a unified key regardless of which prompt was used."""
    if isinstance(d, dict):
        if "rating" in d and "quality" not in d:
            d["quality"] = d.pop("rating")
        if "score" in d and "quality" not in d:
            d["quality"] = d.pop("score")
    return d

def extract_scores(parsed: Dict[str, Any]) -> Dict[str, float]:
    quality = 3.0
    for key in ["quality", "rating", "score"]:
        if key in parsed:
            try:
                quality = float(parsed[key])
                break
            except (ValueError, TypeError):
                continue
    auth = float(parsed.get("authenticity",  3.0))
    corr = float(parsed.get("correspondence", 3.0))
    return {
        "quality":        max(1.0, min(5.0, quality)),
        "authenticity":   max(1.0, min(5.0, auth)),
        "correspondence": max(1.0, min(5.0, corr)),
    }

def extract_degradation(parsed: Dict[str, Any]) -> Dict[str, str]:
    dc  = str(parsed.get("distortion_class", "null")).lower().strip()
    sev = str(parsed.get("severity",         "null")).lower().strip()
    if dc  not in {"noise", "compression", "blur", "darken", "null", "none"}:
        dc  = "null"
    if sev not in {"slight", "moderate", "obvious", "serious", "catastrophic", "null", "none"}:
        sev = "null"
    return {"distortion_class": dc, "severity": sev}

Writing /root/iqa_unified/model_utils.py


In [7]:
%%writefile /root/iqa_unified/rewards.py
import torch
import re

SEVERITY_ORDER = {
    "null": 0, "slight": 1, "moderate": 2,
    "obvious": 3, "serious": 4, "catastrophic": 5,
}


def score_reward(pred: float, gt: float, scale: float = 2.0) -> float:
    """
    Smooth reward using exponential decay.
    Returns value in [0, 1] where 1.0 = perfect match.
    Args:
        pred:  predicted score (e.g., 3.2)
        gt:    ground truth score (e.g., 3.0)
        scale: decay rate — higher = steeper penalty (default 2.0)
    """
    diff = abs(pred - gt)
    return float(torch.exp(torch.tensor(-scale * diff)))


def format_reward(response_text: str) -> float:
    # Q-Insight native format: REASONING: ... ANSWER: {...}
    # This is what the base model currently produces
    if re.search(
        r"REASONING:.*?ANSWER:\s*\{[^}]+\}",
        response_text, re.DOTALL
    ):
        return 1.0

    # Target format: <think>...</think><answer>{...}</answer>
    # This is what the model should converge toward via GRPO
    # Uses [\s\S]*? instead of [^\{\}]* so nested braces don't break it
    pattern = (
        r"^<think>\s*\n"
        r".*?\n"
        r"\s*</think>\s*\n"
        r"<answer>\s*\n"
        r"\{[\s\S]*?\}"
        r"\s*\n"
        r"\s*</answer>\s*$"
    )
    return 1.0 if re.fullmatch(pattern, response_text, re.DOTALL | re.MULTILINE) else 0.0


def degradation_classification_reward(pred_class: str, gt_class: str) -> float:
    pred = pred_class.lower().strip()
    gt   = gt_class.lower().strip()
    if gt in ("null", "none"):
        return 1.0 if pred in ("null", "none") else 0.0
    return 1.0 if pred == gt else 0.0


def severity_perception_reward(pred_sev: str, gt_sev: str, partial: float = 0.5) -> float:
    p = SEVERITY_ORDER.get(pred_sev.lower().strip(), 0)
    g = SEVERITY_ORDER.get(gt_sev.lower().strip(), 0)

    # gt is null — predicting any severity is wrong, not partially right
    if g == 0:
        return 1.0 if p == 0 else 0.0

    # gt has a severity but model predicted null — no credit
    if p == 0:
        return 0.0

    diff = abs(p - g)
    if diff == 0: return 1.0
    if diff == 1: return partial
    return 0.0

Writing /root/iqa_unified/rewards.py


In [8]:
%%writefile /root/iqa_unified/metrics.py
import numpy as np
from scipy import stats
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error, mean_absolute_error
from typing import Dict, List, Any

def compute_srcc(pred, gt):
    if len(pred) < 3: return 0.0
    corr, _ = stats.spearmanr(pred, gt)
    return float(corr) if not np.isnan(corr) else 0.0

def compute_plcc(pred, gt):
    if len(pred) < 3: return 0.0
    pred = np.array(pred); gt = np.array(gt)
    try:
        from scipy.optimize import curve_fit
        def logistic(x, b1, b2, b3, b4):
            return b1 * (0.5 - 1 / (1 + np.exp(b2 * (x - b3)))) + b4
        b1 = gt.max() - gt.min()
        b2 = np.log(9) / (pred.max() - pred.min() + 1e-10)
        popt, _ = curve_fit(logistic, pred, gt, p0=[b1, b2, pred.mean(), gt.mean()], maxfev=10000)
        pred = logistic(pred, *popt)
    except Exception: pass
    corr, _ = stats.pearsonr(pred, gt)
    return float(corr) if not np.isnan(corr) else 0.0

def compute_krcc(pred, gt):
    if len(pred) < 3: return 0.0
    corr, _ = stats.kendalltau(pred, gt)
    return float(corr) if not np.isnan(corr) else 0.0

def compute_rmse(pred, gt): return float(np.sqrt(mean_squared_error(gt, pred)))
def compute_mae(pred, gt):  return float(mean_absolute_error(gt, pred))

def compute_degradation_accuracy(pred_classes, gt_classes):
    mask = [gt.lower() not in ("null","none") for gt in gt_classes]
    pred = [p for p, m in zip(pred_classes, mask) if m]
    gt   = [g for g, m in zip(gt_classes,   mask) if m]
    if not pred: return {"accuracy": 0.0, "macro_f1": 0.0}
    acc = accuracy_score(gt, pred)
    all_cls = sorted(set(gt) | set(pred))
    f1 = f1_score(gt, pred, labels=all_cls, average="macro", zero_division=0) if len(all_cls) > 1 else acc
    return {"accuracy": acc, "macro_f1": f1}

def compute_severity_accuracy(pred_sev, gt_sev):
    mask = [gt.lower() not in ("null","none") for gt in gt_sev]
    pred = [p for p, m in zip(pred_sev, mask) if m]
    gt   = [g for g, m in zip(gt_sev,   mask) if m]
    if not pred: return {"accuracy": 0.0, "macro_f1": 0.0, "adjacent_accuracy": 0.0}
    acc = accuracy_score(gt, pred)
    all_cls = ["slight","moderate","obvious","serious","catastrophic"]
    f1 = f1_score(gt, pred, labels=all_cls, average="macro", zero_division=0)
    order = {s: i for i, s in enumerate(all_cls)}
    adjacent = sum(1 for p, g in zip(pred, gt) if abs(order.get(p,2)-order.get(g,2)) <= 1)
    return {"accuracy": acc, "macro_f1": f1, "adjacent_accuracy": adjacent / len(pred)}

def compute_score_stability(clean, perturbed, threshold=0.5):
    clean = np.array(clean); perturbed = np.array(perturbed)
    changes = np.abs(perturbed - clean)
    return {
        "mean_score_change": float(np.mean(changes)),
        "max_score_change": float(np.max(changes)),
        "significant_change_rate": float(np.mean(changes > threshold)),
        "clean_perturbed_correlation": float(stats.pearsonr(clean, perturbed)[0]) if len(clean) > 2 else 0.0,
        "robustness_score": 1.0 - float(np.mean(changes > threshold)),
    }

def evaluate_predictions(predictions, ground_truths, score_key="quality"):
    results = {}
    pred_scores = [p.get(score_key, 3.0)         for p in predictions]
    gt_scores   = [g.get(f"gt_{score_key}", 3.0) for g in ground_truths]
    valid       = [g.get(f"gt_{score_key}", 0) > 0 for g in ground_truths]
    pv = [p for p, m in zip(pred_scores, valid) if m]
    gv = [g for g, m in zip(gt_scores,   valid) if m]
    if len(pv) >= 3:
        results.update({
            f"srcc_{score_key}": compute_srcc(pv, gv),
            f"plcc_{score_key}": compute_plcc(pv, gv),
            f"krcc_{score_key}": compute_krcc(pv, gv),
            f"rmse_{score_key}": compute_rmse(pv, gv),
            f"mae_{score_key}":  compute_mae(pv, gv),
            f"num_valid_{score_key}": len(pv),
        })
    dist = compute_degradation_accuracy(
        [p.get("distortion_class","null") for p in predictions],
        [g.get("gt_distortion_class","null") for g in ground_truths])
    results.update({f"degradation_{k}": v for k, v in dist.items()})
    sev = compute_severity_accuracy(
        [p.get("severity","null") for p in predictions],
        [g.get("gt_severity","null") for g in ground_truths])
    results.update({f"severity_{k}": v for k, v in sev.items()})
    return results

def format_results(results: Dict[str, Any]) -> str:
    lines = ["\nEvaluation Results:"]
    for k, v in results.items():
        lines.append(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
    return "\n".join(lines)

Writing /root/iqa_unified/metrics.py


In [9]:
%%writefile /root/iqa_unified/adversarial.py
import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image
from torchvision import transforms
from model_utils import generate_response, parse_answer, extract_scores
from prompts import build_single_image_message, SCORE_ONLY_PROMPT

class SurrogateIQAModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.features = torch.nn.Sequential(
            torch.nn.Conv2d(3, 64, 3, padding=1), torch.nn.ReLU(), torch.nn.MaxPool2d(2),
            torch.nn.Conv2d(64, 128, 3, padding=1), torch.nn.ReLU(), torch.nn.MaxPool2d(2),
            torch.nn.Conv2d(128, 256, 3, padding=1), torch.nn.ReLU(),
            torch.nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.regressor = torch.nn.Sequential(
            torch.nn.Linear(256, 128), torch.nn.ReLU(), torch.nn.Dropout(0.2),
            torch.nn.Linear(128, 1), torch.nn.Sigmoid(),
        )
    def forward(self, x):
        x = self.features(x)
        return self.regressor(x.view(x.size(0), -1)) * 4 + 1

class AdversarialAttacker:
    def __init__(self, model, processor, config, surrogate_epochs=5):
        self.model = model; self.processor = processor; self.config = config
        self.device = next(model.parameters()).device
        self.surrogate = None; self.surrogate_epochs = surrogate_epochs

    def train_surrogate(self, images, num_samples=100):
        print("Training surrogate...")
        self.surrogate = SurrogateIQAModel().to(self.device)
        optimizer = torch.optim.Adam(self.surrogate.parameters(), lr=1e-3)
        train_images = images[:num_samples] if len(images) > num_samples else images
        transform = transforms.Compose([
            transforms.Resize(224), transforms.CenterCrop(224), transforms.ToTensor(),
            transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ])
        vlm_scores, tensors = [], []
        for img in train_images:
            messages = build_single_image_message(img, SCORE_ONLY_PROMPT)
            response = generate_response(self.model, self.processor, messages, max_new_tokens=64, do_sample=False)
            vlm_scores.append(extract_scores(parse_answer(response)).get("quality", 3.0))
            tensors.append(transform(img))
        vlm_scores = torch.tensor(vlm_scores, device=self.device)
        tensors = torch.stack(tensors).to(self.device)
        self.surrogate.train()
        for _ in range(self.surrogate_epochs):
            optimizer.zero_grad()
            loss = F.mse_loss(self.surrogate(tensors).squeeze(), vlm_scores)
            loss.backward(); optimizer.step()
        self.surrogate.eval()
        print(f"Surrogate trained. MSE: {loss.item():.4f}")

    def pgd_attack_surrogate(self, pil_image, epsilon=8/255, alpha=2/255, num_steps=20):
        if self.surrogate is None: raise ValueError("Train surrogate first")
        transform = transforms.Compose([
            transforms.Resize(224), transforms.CenterCrop(224), transforms.ToTensor(),
            transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ])
        inv_transform = transforms.Compose([
            transforms.Normalize(mean=[0.,0.,0.], std=[1/0.229,1/0.224,1/0.225]),
            transforms.Normalize(mean=[-0.485,-0.456,-0.406], std=[1.,1.,1.]),
        ])
        x = transform(pil_image).unsqueeze(0).to(self.device)
        x_orig = x.clone()
        clean_score = extract_scores(parse_answer(
            generate_response(self.model, self.processor,
                              build_single_image_message(pil_image, SCORE_ONLY_PROMPT),
                              max_new_tokens=64, do_sample=False)
        )).get("quality", 3.0)
        delta = torch.zeros_like(x, requires_grad=True)
        for _ in range(num_steps):
            self.surrogate.zero_grad()
            self.surrogate(x + delta).sum().backward()
            grad = delta.grad.detach()
            delta.data = torch.clamp(delta.data + alpha * grad.sign(), -epsilon, epsilon)
            delta.data = torch.clamp(x_orig + delta.data, 0, 1) - x_orig
            delta.grad.zero_()
        adv_img = transforms.ToPILImage()(inv_transform((x_orig+delta).squeeze(0)).clamp(0,1))
        adv_score = extract_scores(parse_answer(
            generate_response(self.model, self.processor,
                              build_single_image_message(adv_img, SCORE_ONLY_PROMPT),
                              max_new_tokens=64, do_sample=False)
        )).get("quality", 3.0)
        return adv_img, {
            "clean_score": clean_score, "perturbed_score": adv_score,
            "score_change": abs(adv_score - clean_score),
            "perturbation_l_inf": delta.abs().max().item(),
            "success": abs(adv_score - clean_score) > self.config.adversarial.score_change_threshold,
        }

    def evaluate_robustness(self, dataset, max_samples=100):
        print(f"\nEvaluating adversarial robustness ({max_samples} samples)...")
        all_images = [dataset[i]["image"] for i in range(min(len(dataset), max_samples*2))]
        self.train_surrogate(all_images, num_samples=min(100, len(all_images)))
        results = []
        for i in range(min(max_samples, len(dataset))):
            _, info = self.pgd_attack_surrogate(
                dataset[i]["image"], epsilon=self.config.adversarial.epsilon,
                alpha=self.config.adversarial.alpha, num_steps=self.config.adversarial.num_steps)
            results.append(info)
            if (i+1) % 10 == 0: print(f"  Attacked {i+1}/{min(max_samples, len(dataset))}")
        mean_change  = np.mean([r["score_change"] for r in results])
        success_rate = np.mean([r["success"]      for r in results])
        return {
            "mean_score_change": float(mean_change),
            "attack_success_rate": float(success_rate),
            "robustness_score": 1.0 - float(success_rate),
            "num_samples": len(results),
        }

Writing /root/iqa_unified/adversarial.py


In [10]:
%%writefile /root/iqa_unified/grpo_trainer.py
import torch, gc, os, json, time
from torch.utils.data import DataLoader
from typing import Optional, Dict, List, Tuple
from dataclasses import dataclass
from model_utils import (generate_response, parse_answer, extract_scores,
                         extract_degradation, clear_gpu_memory)
from prompts import (PREFERENCE_PROMPT, SCORE_DEGRADATION_PROMPT,
                     SCORE_ONLY_PROMPT, build_single_image_message)
from rewards import (format_reward, score_reward,
                     degradation_classification_reward, severity_perception_reward)
from qwen_vl_utils import process_vision_info
from peft import PeftModel
import numpy as np


@dataclass
class GRPOTrainingState:
    step: int = 0
    epoch: int = 0
    best_reward: float = -float("inf")


class GRPOTrainer:
    def __init__(self, model, processor, config,
                 train_loader: DataLoader, val_loader: Optional[DataLoader] = None):
        self.processor    = processor
        self.config       = config
        self.train_loader = train_loader
        self.val_loader   = val_loader
        self.model        = model
        self.device       = next(model.parameters()).device
        self.state        = GRPOTrainingState()

        self.global_step  = 0
        self.total_updates = (len(train_loader) * config.grpo.num_epochs
                              // config.grpo.gradient_accumulation_steps)
        self.accum_counter = 0

        trainable = [p for p in self.model.parameters() if p.requires_grad]
        print(f"Trainable parameters: {sum(p.numel() for p in trainable):,}")

        self.optimizer = torch.optim.AdamW(
            trainable, lr=config.grpo.learning_rate, weight_decay=config.grpo.weight_decay)
        warmup_steps = int(self.total_updates * config.grpo.warmup_ratio)
        self.scheduler = torch.optim.lr_scheduler.LinearLR(
            self.optimizer, start_factor=0.1, total_iters=max(warmup_steps, 1))
        os.makedirs(config.checkpoint_dir, exist_ok=True)

    # ------------------------------------------------------------------
    # Task-specific prompt selection
    # ------------------------------------------------------------------
    def _get_prompt_for_sample(self, gt_dict: Dict) -> str:
        has_auth   = gt_dict.get("gt_authenticity",   -1.0) > 0
        has_degrad = gt_dict.get("gt_distortion_class", "null") != "null"
        has_qual   = gt_dict.get("gt_quality",          0.0) > 0
        if has_auth:
            return PREFERENCE_PROMPT
        if has_degrad:
            return SCORE_DEGRADATION_PROMPT
        if has_qual:
            return SCORE_ONLY_PROMPT
        return PREFERENCE_PROMPT

    # ------------------------------------------------------------------
    # Per-token logprobs
    # Input:  full sequence ids (prompt + completion), shape (1, P+C)
    # Output: per-token logps,                         shape (1, P+C-1)
    # ------------------------------------------------------------------
    def _get_per_token_logps(self, model, input_ids, attention_mask,
                              pixel_values, image_grid_thw) -> torch.Tensor:
        amp_dtype = next(self.model.parameters()).dtype
        with torch.cuda.amp.autocast(dtype=amp_dtype):
            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values,
                image_grid_thw=image_grid_thw,
                return_dict=True,
            ).logits                             # (1, L, V)

        logits     = logits[:, :-1, :]           # (1, L-1, V)
        target_ids = input_ids[:, 1:]            # (1, L-1)

        log_probs = logits.float().log_softmax(dim=-1)
        per_token_logps = []
        for logits_row, ids_row in zip(log_probs, target_ids):
            token_lp = logits_row.gather(1, ids_row.unsqueeze(1)).squeeze(1)
            per_token_logps.append(token_lp)
        return torch.stack(per_token_logps)      # (1, L-1)

    # ------------------------------------------------------------------
    # Candidate generation
    # ------------------------------------------------------------------
    def generate_candidates(self, image, text_prompt: str,
                             num_candidates: int) -> List[Dict]:
        messages = build_single_image_message(image, text_prompt)
        text     = self.processor.apply_chat_template(messages, tokenize=False,
                                                       add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = self.processor(text=[text], images=image_inputs, videos=video_inputs,
                                padding=True, return_tensors="pt")

        pixel_values   = inputs["pixel_values"].to(self.device)
        input_ids      = inputs["input_ids"].to(self.device)
        attention_mask = inputs["attention_mask"].to(self.device)
        image_grid_thw = inputs.get("image_grid_thw")
        if image_grid_thw is not None:
            image_grid_thw = image_grid_thw.to(self.device)

        prompt_length = input_ids.shape[1]
        eos_id        = self.processor.tokenizer.eos_token_id
        candidates    = []

        # FIX 1: minimum temp raised to 0.8 — temps below this collapse to
        # identical greedy outputs, producing 3+ tied rewards and zero
        # useful gradient signal from those candidates
        temp_range = [0.8, 1.0, 1.3, 1.6, 2.0, 2.5, 3.0, 3.5]

        for idx in range(num_candidates):
            temp = temp_range[idx % len(temp_range)]

            with torch.no_grad():
                gen_out = self.model.generate(
                    input_ids=input_ids,
                    pixel_values=pixel_values,
                    attention_mask=attention_mask,
                    image_grid_thw=image_grid_thw,
                    max_new_tokens=self.config.model.max_new_tokens,
                    do_sample=self.config.model.do_sample,
                    temperature=temp,
                    top_p=self.config.model.top_p,
                    top_k=50 if temp > 1.5 else 0,
                    num_return_sequences=1,
                    use_cache=True,
                )

            full_ids   = gen_out                               # (1, P+C)
            completion = full_ids[:, prompt_length:]          # (1, C)

            is_eos = (completion[0] == eos_id)
            if is_eos.any():
                first_eos = is_eos.nonzero(as_tuple=False)[0, 0].item()
                comp_mask = torch.zeros(completion.shape[1], dtype=torch.long,
                                        device=self.device)
                comp_mask[:first_eos + 1] = 1
            else:
                comp_mask = torch.ones(completion.shape[1], dtype=torch.long,
                                       device=self.device)
            comp_mask = comp_mask.unsqueeze(0)                # (1, C)
            full_mask = torch.cat([attention_mask, comp_mask], dim=1)  # (1, P+C)

            resp_text = self.processor.batch_decode(completion, skip_special_tokens=True)[0]
            parsed    = parse_answer(resp_text)

            # FIX 2: capture reference logprobs under eval() so dropout is
            # disabled — training mode made old_logps stochastic, which
            # corrupted the PPO ratio for every subsequent update
            self.model.eval()
            with torch.no_grad():
                old_logps = self._get_per_token_logps(
                    self.model, full_ids, full_mask,
                    pixel_values, image_grid_thw
                )[:, prompt_length - 1:]
            self.model.train()

            candidates.append({
                "response_text":         resp_text,
                "full_ids":              full_ids,
                "full_mask":             full_mask,
                "completion_mask":       comp_mask,
                "prompt_length":         prompt_length,
                "pixel_values":          pixel_values,
                "image_grid_thw":        image_grid_thw,
                "old_per_token_logps":   old_logps,
                "parsed":                parsed,
                "predicted_scores":      extract_scores(parsed),
                "predicted_degradation": extract_degradation(parsed),
                "reward":                0.0,
            })

        clear_gpu_memory()
        return candidates

    # ------------------------------------------------------------------
    # Content reward
    # ------------------------------------------------------------------
    def _compute_content_reward(self, c: Dict, gt_scores: Dict,
                                gt_degrad: Dict) -> float:
        total = 0.0
        norm  = 0.0
        cfg   = self.config.rewards

        total += cfg.quality_score_weight * score_reward(
            c["predicted_scores"].get("quality", 3.0),
            gt_scores["quality"],
            scale=cfg.score_decay_scale)
        norm += cfg.quality_score_weight

        if gt_scores["authenticity"] > 0:
            total += cfg.authenticity_score_weight * score_reward(
                c["predicted_scores"].get("authenticity", 3.0),
                gt_scores["authenticity"],
                scale=cfg.score_decay_scale)
            norm += cfg.authenticity_score_weight

        if gt_scores["correspondence"] > 0:
            total += cfg.correspondence_score_weight * score_reward(
                c["predicted_scores"].get("correspondence", 3.0),
                gt_scores["correspondence"],
                scale=cfg.score_decay_scale)
            norm += cfg.correspondence_score_weight

        if gt_degrad["distortion_class"] != "null":
            class_correct = degradation_classification_reward(
                c["predicted_degradation"]["distortion_class"],
                gt_degrad["distortion_class"])
            total += cfg.degradation_class_weight * class_correct
            norm  += cfg.degradation_class_weight

            if gt_degrad["severity"] != "null":
                sev_r = severity_perception_reward(
                    c["predicted_degradation"]["severity"],
                    gt_degrad["severity"]) if class_correct == 1.0 else 0.0
                total += cfg.severity_class_weight * sev_r
                norm  += cfg.severity_class_weight

        return (total / norm) if norm > 0 else 0.0

    # ------------------------------------------------------------------
    # Reward computation
    # ------------------------------------------------------------------
    def compute_rewards_for_group(self, candidates: List[Dict],
                                  ground_truth: Dict) -> torch.Tensor:
        gt_scores = {
            "quality":        ground_truth.get("gt_quality",        3.0),
            "authenticity":   ground_truth.get("gt_authenticity",  -1.0),
            "correspondence": ground_truth.get("gt_correspondence", -1.0),
        }
        gt_degrad = {
            "distortion_class": ground_truth.get("gt_distortion_class", "null"),
            "severity":         ground_truth.get("gt_severity",         "null"),
        }
        rewards = []
        for c in candidates:
            fmt_r   = format_reward(c["response_text"])
            content = self._compute_content_reward(c, gt_scores, gt_degrad)
            final   = 0.2 * fmt_r + 0.8 * content
            c["reward"] = final
            rewards.append(final)

            if os.getenv("DEBUG_MODE") == "true":
                print(f"  [DEBUG] fmt={fmt_r:.1f} content={content:.3f} "
                      f"total={final:.3f} | "
                      f"pred_q={c['predicted_scores'].get('quality',0):.2f} "
                      f"gt_q={gt_scores['quality']:.2f}")

        return torch.tensor(rewards, dtype=torch.float32, device=self.device)

    # ------------------------------------------------------------------
    # Advantage normalisation with clipping
    # FIX 3: skip update when all candidates score similarly — previously
    # std < 1e-8 subtracted the mean but still ran the update, amplifying
    # floating-point noise into fake large advantages; threshold raised to
    # 0.02 to cover near-degenerate groups (step 0 had spread of 0.011)
    # ------------------------------------------------------------------
    def _compute_advantages(self, rewards: torch.Tensor) -> torch.Tensor:
        mean = rewards.mean()
        std  = rewards.std()
        if std < 0.005:
            # No meaningful signal in this group — zero advantages skip the
            # backward pass effectively (loss → 0, no parameter update)
            return torch.zeros_like(rewards)
        adv = (rewards - mean) / (std + 1e-4)
        return torch.clamp(adv, -2.0, 2.0)

    # ------------------------------------------------------------------
    # GRPO update
    # FIX 4: scale denominator is now only gradient_accumulation_steps.
    # The old denominator (accum_steps * num_iterations) assumed a single
    # optimizer step would accumulate gradients from all iterations, but
    # each grpo_update call with do_step=True steps independently, so the
    # extra num_iterations factor under-scaled every gradient by 2×.
    # ------------------------------------------------------------------
    def grpo_update(self, candidates: List[Dict],
                    advantages: torch.Tensor,
                    do_step: bool = False) -> Dict:
        total_loss = 0.0   # plain float — no need for a GPU tensor here
        num_valid  = 0

        for i, c in enumerate(candidates):
            adv             = advantages[i]
            comp_mask_float = c["completion_mask"].float()
            token_count     = comp_mask_float.sum().clamp(min=1)

            # Length-normalised advantage
            adv_normalized      = adv / torch.sqrt(token_count)
            old_per_token_logps = c["old_per_token_logps"]

            per_token_logps = self._get_per_token_logps(
                self.model,
                c["full_ids"],
                c["full_mask"],
                c["pixel_values"],
                c["image_grid_thw"],
            )
            prompt_len      = c["prompt_length"]
            per_token_logps = per_token_logps[:, prompt_len - 1:]  # (1, C)

            # PPO-style clipped surrogate loss
            ratio   = torch.exp(per_token_logps - old_per_token_logps)
            clipped = torch.clamp(ratio,
                                  1 - self.config.grpo.clip_range,
                                  1 + self.config.grpo.clip_range)
            per_token_loss = -torch.min(ratio * adv_normalized,
                                        clipped * adv_normalized)

            loss   = (per_token_loss * comp_mask_float).sum() / token_count
            scale  = self.config.grpo.gradient_accumulation_steps   # FIX 4
            scaled = loss / scale
            scaled.backward()
            total_loss += loss.item()
            num_valid  += 1

        if do_step and num_valid > 0:
            torch.nn.utils.clip_grad_norm_(
                [p for p in self.model.parameters() if p.requires_grad],
                self.config.grpo.max_grad_norm)
            self.optimizer.step()
            self.optimizer.zero_grad()
            self.scheduler.step()
            self.global_step += 1

        return {
            "policy_loss": total_loss / max(num_valid, 1),
            "num_valid":   num_valid,
        }

    # ------------------------------------------------------------------
    # Checkpointing
    # ------------------------------------------------------------------
    def save_checkpoint(self, epoch: int, step: int, is_best: bool = False):
        payload = {
            "epoch":                epoch,
            "step":                 step,
            "global_step":          self.global_step,
            "optimizer_state_dict": self.optimizer.state_dict(),
            "scheduler_state_dict": self.scheduler.state_dict(),
            "best_reward":          self.state.best_reward,
        }
        if self.config.model.use_lora:
            payload["lora_state_dict"] = {k: v for k, v in self.model.state_dict().items()
                                          if "lora" in k}
        else:
            payload["model_state_dict"] = self.model.state_dict()
        ckpt   = os.path.join(self.config.checkpoint_dir, f"ckpt_e{epoch}_s{step}.pt")
        latest = os.path.join(self.config.checkpoint_dir, "ckpt_latest.pt")
        torch.save(payload, ckpt)
        torch.save(payload, latest)
        if is_best:
            torch.save(payload, os.path.join(self.config.checkpoint_dir, "ckpt_best.pt"))
        self._cleanup_old_checkpoints(keep=2)

    def _cleanup_old_checkpoints(self, keep: int = 2):
        ckpts = sorted(
            [f for f in os.listdir(self.config.checkpoint_dir)
             if f.startswith("ckpt_e") and f.endswith(".pt")],
            key=lambda x: os.path.getmtime(os.path.join(self.config.checkpoint_dir, x)))
        for old in ckpts[:-keep]:
            os.remove(os.path.join(self.config.checkpoint_dir, old))

    def load_checkpoint(self, path: str):
        print(f"Loading checkpoint from {path}")
        payload = torch.load(path, map_location=self.device)
        key = "lora_state_dict" if "lora_state_dict" in payload else "model_state_dict"
        self.model.load_state_dict(payload[key], strict=False)
        self.optimizer.load_state_dict(payload["optimizer_state_dict"])
        self.scheduler.load_state_dict(payload["scheduler_state_dict"])
        self.state.best_reward = payload.get("best_reward", -float("inf"))
        self.global_step = payload.get("global_step", 0)
        print(f"Resumed: epoch={payload['epoch']}, step={payload['step']}, "
              f"global_step={self.global_step}")
        return payload["epoch"], payload["step"]

    # ------------------------------------------------------------------
    # Training loop
    # ------------------------------------------------------------------
    def train(self) -> Dict:
        start_epoch = 0
        if self.config.resume_from and os.path.exists(self.config.resume_from):
            start_epoch, _ = self.load_checkpoint(self.config.resume_from)
        print("=" * 60)
        print(f"GRPO TRAINING  |  {self.config.mixed_precision}  |  "
              f"lr={self.config.grpo.learning_rate}")
        print(f"Epochs: {self.config.grpo.num_epochs}  "
              f"Groups: {self.config.grpo.num_groups}  "
              f"Iterations: {self.config.grpo.num_iterations}  "
              f"4-bit: {self.config.model.use_4bit}  "
              f"LoRA: {self.config.model.use_lora}")
        print("GRPO — wide temperature diversity, smooth rewards, no entropy term")
        print("=" * 60)
        t0 = time.time()
        for epoch in range(start_epoch, self.config.grpo.num_epochs):
            self.state.epoch = epoch
            self._train_epoch(epoch)
            self.save_checkpoint(epoch, self.state.step)
            if self.val_loader:
                self._evaluate()
        return {"best_reward": self.state.best_reward,
                "total_time":  time.time() - t0}

    def _train_epoch(self, epoch_num: int):
        self.model.train()
        history     = {"rewards": [], "loss": []}
        total_steps = len(self.train_loader)
        epoch_start = time.time()

        for step, batch in enumerate(self.train_loader):
            self.state.step = step
            step_start      = time.time()

            for i in range(len(batch["images"])):
                self.accum_counter += 1
                do_step = (self.accum_counter % self.config.grpo.gradient_accumulation_steps == 0)

                image   = batch["images"][i]
                gt_dict = {
                    "gt_quality":          batch["gt_quality"][i].item(),
                    "gt_authenticity":     batch["gt_authenticity"][i].item(),
                    "gt_correspondence":   batch["gt_correspondence"][i].item(),
                    "gt_distortion_class": batch["gt_distortion_class"][i],
                    "gt_severity":         batch["gt_severity"][i],
                }

                prompt     = self._get_prompt_for_sample(gt_dict)
                candidates = self.generate_candidates(image, prompt,
                                                      self.config.grpo.num_groups)
                rewards    = self.compute_rewards_for_group(candidates, gt_dict)
                advantages = self._compute_advantages(rewards)

                if step < 3 and i == 0:
                    print(f"  [DEBUG step {step}] rewards: {[f'{r:.4f}' for r in rewards.tolist()]}")
                    print(f"  [DEBUG step {step}] advantages: {[f'{a:.4f}' for a in advantages.tolist()]}")
                    print(f"  [DEBUG step {step}] reward_std: {rewards.std().item():.4f}  "
                          f"skipped={'yes' if rewards.std().item() < 0.02 else 'no'}")

                # FIX 5: only call optimizer.step() on the LAST iteration.
                # Previously do_step=True on every iteration caused
                # num_iterations independent optimizer steps per accumulation
                # cycle, doubling the effective LR and causing the reward
                # oscillation / policy collapse seen in the training logs.
                last_metrics = {}
                for iter_idx in range(self.config.grpo.num_iterations):
                    do_update = do_step and (iter_idx == self.config.grpo.num_iterations - 1)
                    last_metrics = self.grpo_update(candidates, advantages,
                                                    do_step=do_update)

                history["rewards"].append(rewards.mean().item())
                history["loss"].append(last_metrics["policy_loss"])

            if (step + 1) % self.config.save_steps == 0:
                rolling_mean = np.mean(history["rewards"][-50:]) if history["rewards"] else 0.0
                is_best = rolling_mean > self.state.best_reward
                if is_best:
                    self.state.best_reward = rolling_mean
                self.save_checkpoint(epoch_num, step + 1, is_best=is_best)

            elapsed       = time.time() - epoch_start
            secs_per_step = elapsed / (step + 1)
            eta_sec       = secs_per_step * (total_steps - step - 1)
            eta_str       = time.strftime("%H:%M:%S", time.gmtime(eta_sec))
            pct           = (step + 1) / total_steps * 100
            tail          = slice(-10, None)
            avg_r  = np.mean(history["rewards"][tail]) if history["rewards"] else 0.0
            avg_l  = np.mean(history["loss"][tail])    if history["loss"]    else 0.0
            mem0   = torch.cuda.memory_allocated(0) / 1e9
            mem1   = torch.cuda.memory_allocated(1) / 1e9 if torch.cuda.device_count() > 1 else 0

            print(
                f"[{pct:5.1f}%] Ep{epoch_num} {step+1:4d}/{total_steps} | "
                f"R={avg_r:.3f} L={avg_l:.6f} | "
                f"GPU0={mem0:.1f}GB GPU1={mem1:.1f}GB | "
                f"step={time.time()-step_start:.1f}s ETA={eta_str}", flush=True)

        total_epoch_time = time.time() - epoch_start
        print(f"\nEpoch {epoch_num} finished in {total_epoch_time/60:.1f} min "
              f"({total_epoch_time/total_steps:.1f}s/step avg)")

    # ------------------------------------------------------------------
    # Validation
    # ------------------------------------------------------------------
    @torch.no_grad()
    def _evaluate(self):
        self.model.eval()
        rewards = []
        for batch in self.val_loader:
            for i in range(min(len(batch["images"]), 10)):
                gt_dict = {
                    "gt_quality":          batch["gt_quality"][i].item(),
                    "gt_authenticity":     batch["gt_authenticity"][i].item(),
                    "gt_correspondence":   batch["gt_correspondence"][i].item(),
                    "gt_distortion_class": batch["gt_distortion_class"][i],
                    "gt_severity":         batch["gt_severity"][i],
                }
                prompt   = self._get_prompt_for_sample(gt_dict)
                messages = build_single_image_message(batch["images"][i], prompt)
                response = generate_response(self.model, self.processor, messages,
                                             max_new_tokens=self.config.model.max_new_tokens,
                                             do_sample=False)
                parsed  = parse_answer(response)
                fmt_r   = format_reward(response)
                gt_scores = {
                    "quality":        gt_dict["gt_quality"],
                    "authenticity":   gt_dict["gt_authenticity"],
                    "correspondence": gt_dict["gt_correspondence"],
                }
                gt_degrad = {
                    "distortion_class": gt_dict["gt_distortion_class"],
                    "severity":         gt_dict["gt_severity"],
                }
                c = {
                    "predicted_scores":      extract_scores(parsed),
                    "predicted_degradation": extract_degradation(parsed),
                }
                content = self._compute_content_reward(c, gt_scores, gt_degrad)
                rewards.append(0.2 * fmt_r + 0.8 * content)

        mean_reward = np.mean(rewards) if rewards else 0.0
        if mean_reward > self.state.best_reward:
            self.state.best_reward = mean_reward
        print(f"Val reward: {mean_reward:.4f}  (best: {self.state.best_reward:.4f})")

Writing /root/iqa_unified/grpo_trainer.py


In [11]:
%%writefile /root/iqa_unified/evaluate.py
import torch, json, gc
from pathlib import Path
from typing import Dict, List, Any
from model_utils import (generate_response, parse_answer, extract_scores,
                         extract_degradation, parse_thinking, clear_gpu_memory)
from prompts import build_single_image_message, get_prompt_for_dataset
from iqa_datasets import create_dataset, BaseIQADataset
from metrics import evaluate_predictions, format_results
from adversarial import AdversarialAttacker
import numpy as np


class IQAEvaluator:
    def __init__(self, model, processor, config):
        self.model, self.processor, self.config = model, processor, config
        self.device = next(model.parameters()).device

    def evaluate_dataset(self, dataset: BaseIQADataset, dataset_name="",
                         max_samples=-1, score_key="quality"):
        self.model.eval()
        n = min(max_samples, len(dataset)) if max_samples > 0 else len(dataset)
        print(f"\nEvaluating {dataset_name} ({n} samples)...")

        # Use dataset-specific prompt — not UNIFIED_ASSESSMENT_PROMPT
        prompt = get_prompt_for_dataset(dataset_name)

        predictions, ground_truths, reasonings = [], [], []
        for i in range(n):
            sample   = dataset[i]
            messages = build_single_image_message(sample["image"], prompt)
            response = generate_response(self.model, self.processor, messages,
                                         max_new_tokens=256, do_sample=False)
            parsed = parse_answer(response)
            predictions.append({**extract_scores(parsed), **extract_degradation(parsed)})
            ground_truths.append({f"gt_{k}": sample[f"gt_{k}"]
                                  for k in ["quality", "authenticity", "correspondence",
                                            "distortion_class", "severity"]})
            reasonings.append(parse_thinking(response))
            if (i + 1) % 10 == 0: clear_gpu_memory()

        results = evaluate_predictions(predictions, ground_truths, score_key)
        results["dataset_name"] = dataset_name or dataset.dataset_name
        results["num_samples"]  = n
        if self.config.eval.save_predictions:
            self._save_predictions(predictions, ground_truths, reasonings, dataset_name)
        return results

    def run_full_evaluation(self, dataset_configs, base_dir="/root",
                             max_samples=-1, run_adversarial=False):
        all_results = {"on_manifold": {}, "adversarial": {}}
        for cfg in dataset_configs:
            try:
                ds  = create_dataset(cfg["name"], cfg.get("dir", base_dir+"/"+cfg["name"]), max_samples)
                res = self.evaluate_dataset(ds, cfg["name"], max_samples)
                all_results["on_manifold"][cfg["name"]] = res
                print(self._format_results(cfg["name"], res))
            except Exception as e:
                print(f"Warning: {cfg['name']} failed: {e}")
                all_results["on_manifold"][cfg["name"]] = {"error": str(e)}
        if run_adversarial and dataset_configs:
            try:
                print("\n" + "="*60 + "\nADVERSARIAL EVALUATION\n" + "="*60)
                ds_name = dataset_configs[0]["name"]
                ds = create_dataset(ds_name, dataset_configs[0].get("dir",
                                    base_dir+"/"+ds_name), max_samples=200)
                attacker    = AdversarialAttacker(self.model, self.processor, self.config)
                adv_results = attacker.evaluate_robustness(ds, max_samples=min(100, len(ds)))
                all_results["adversarial"] = adv_results
                print(f"  Mean Score Change:   {adv_results['mean_score_change']:.4f}")
                print(f"  Attack Success Rate: {adv_results['attack_success_rate']:.4f}")
                print(f"  Robustness Score:    {adv_results['robustness_score']:.4f}")
            except Exception as e:
                print(f"Adversarial evaluation failed: {e}")
                all_results["adversarial"] = {"error": str(e)}
        all_results["interpretability"] = {}
        out_path = Path(self.config.eval.output_dir) / "evaluation_results.json"
        out_path.parent.mkdir(parents=True, exist_ok=True)
        with open(out_path, "w") as f:
            json.dump(all_results, f, indent=2, default=str)
        return all_results

    def _format_results(self, dataset_name: str, res: Dict) -> str:
        """Print only metrics relevant to the dataset's GT fields."""
        lines = [f"\n{dataset_name} ({res.get('num_samples', '?')} samples):"]

        # Quality metrics — always shown
        lines.append(f"  SRCC:  {res.get('srcc_quality', 0):.4f}")
        lines.append(f"  PLCC:  {res.get('plcc_quality', 0):.4f}")
        lines.append(f"  KRCC:  {res.get('krcc_quality', 0):.4f}")
        lines.append(f"  RMSE:  {res.get('rmse_quality', 0):.4f}")
        lines.append(f"  MAE:   {res.get('mae_quality',  0):.4f}")

        # Authenticity — only aigciqa2023 has GT
        if res.get("num_valid_authenticity", 0) > 0:
            lines.append(f"  SRCC Auth: {res.get('srcc_authenticity', 0):.4f}")
            lines.append(f"  PLCC Auth: {res.get('plcc_authenticity', 0):.4f}")

        # Correspondence — only aigciqa2023 has GT
        if res.get("num_valid_correspondence", 0) > 0:
            lines.append(f"  SRCC Corr: {res.get('srcc_correspondence', 0):.4f}")
            lines.append(f"  PLCC Corr: {res.get('plcc_correspondence', 0):.4f}")

        # Degradation — only kadid10k has GT
        if res.get("degradation_accuracy", 0) > 0:
            lines.append(f"  Deg Acc:   {res.get('degradation_accuracy', 0):.4f}")
            lines.append(f"  Deg F1:    {res.get('degradation_macro_f1', 0):.4f}")
            lines.append(f"  Sev Acc:   {res.get('severity_accuracy', 0):.4f}")
            lines.append(f"  Sev Adj:   {res.get('severity_adjacent_accuracy', 0):.4f}")

        return "\n".join(lines)

    def _save_predictions(self, predictions, ground_truths, reasonings, name):
        out_dir = Path(self.config.eval.output_dir) / "predictions"
        out_dir.mkdir(parents=True, exist_ok=True)
        data = [{"prediction": p, "ground_truth": gt, "reasoning": r}
                for p, gt, r in zip(predictions, ground_truths, reasonings)]
        with open(out_dir / f"{name}_predictions.json", "w") as f:
            json.dump(data, f, indent=2)

Writing /root/iqa_unified/evaluate.py


In [12]:
!mkdir -p /root/aigciqa2023/images

# Download zip
!wget -q --show-progress \
    https://huggingface.co/datasets/IntMeGroup/AIGCIQA2023/resolve/main/allimg.zip \
    -O /tmp/allimg.zip

# Download CSV
from huggingface_hub import hf_hub_download
import shutil, os

csv_src = hf_hub_download(
    repo_id="IntMeGroup/AIGCIQA2023",
    filename="AIGIQA2023+.csv",
    repo_type="dataset",
)
shutil.copy(csv_src, "/root/aigciqa2023/AIGIQA2023+.csv")
print(f"CSV size: {os.path.getsize('/root/aigciqa2023/AIGIQA2023+.csv'):,} bytes")

# Unzip images
!unzip -q /tmp/allimg.zip -d /tmp/allimg && \
    mv /tmp/allimg/allimg/*.png /root/aigciqa2023/images/ && \
    rm -rf /tmp/allimg /tmp/allimg.zip

print("Done")

/tmp/allimg.zip     100%[===================>] 847.93M  90.9MB/s    in 9.5s    


AIGIQA2023+.csv:   0%|          | 0.00/290k [00:00<?, ?B/s]

CSV size: 290,305 bytes
Done


In [13]:
%uv pip install pyiqa -q
from pyiqa.data.dataset_api import load_dataset
import shutil, os, pandas as pd

# Download
dataset = load_dataset("kadid10k", "/root/kadid10k", download=True)
print(f"KADID-10k loaded: {len(dataset)} samples")

# Move contents up if nested
src = "/root/kadid10k/kadid10k"
dst = "/root/kadid10k"
if os.path.exists(src):
    for item in os.listdir(src):
        shutil.move(os.path.join(src, item), os.path.join(dst, item))
    os.rmdir(src)
    print("Moved nested directory up.")

# Rename dmos.csv columns to match loader expectations
csv_path = dst + "/dmos.csv"
target_path = dst + "/image_labeled_by_per_noise.csv"

if os.path.exists(csv_path) and not os.path.exists(target_path):
    df = pd.read_csv(csv_path)
    # Rename columns to match loader
    df = df.rename(columns={"dist_img": "image"})
    # Add noise column from filename: I01_01_01.png → type 01
    df["noise"] = df["image"].apply(lambda x: int(x.split("_")[1]))
    df.to_csv(target_path, index=False)
    print(f"Created image_labeled_by_per_noise.csv with {len(df)} rows")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(3).to_string())

print("\nFinal contents of /root/kadid10k:")
print(os.listdir(dst))

Note: you may need to restart the kernel to use updated packages.
Loading dataset kadid10k from /root/kadid10k ...


Fetching 20 files:   0%|          | 0/20 [00:00<?, ?it/s]

meta_info_CSIQDataset.csv: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

meta_info_GFIQADataset.csv: 0.00B [00:00, ?B/s]

...

meta_info_TID2013Dataset.csv: 0.00B [00:00, ?B/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

kadid10k.tgz:   0%|          | 0.00/3.07G [00:00<?, ?B/s]

Extracting TAR/TGZ: 100%|█████████████████████████████████████████████████| 3.07G/3.07G [00:33<00:00, 91.7MB/s]


KADID-10k loaded: 10125 samples
Moved nested directory up.
Created image_labeled_by_per_noise.csv with 10125 rows
Columns: ['image', 'ref_img', 'dmos', 'var', 'noise']
           image  ref_img  dmos    var  noise
0  I01_01_01.png  I01.png  4.57  0.496      1
1  I01_01_02.png  I01.png  4.33  0.869      1
2  I01_01_03.png  I01.png  2.67  0.789      1

Final contents of /root/kadid10k:
['meta_info', '.cache', 'kadid10k.tgz', '.DS_Store', 'images', 'dmos.csv', 'image_labeled_by_per_noise.csv']


In [14]:
from transformers.models.qwen2_5_vl.modeling_qwen2_5_vl import (
    Qwen2_5_VLVisionFlashAttention2,
)
from flash_attn import flash_attn_varlen_func
import torch
from typing import Optional, Tuple

def rotate_half(x):
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb_custom(q, k, cos, sin):
    # q, k: (seq_len, num_heads, head_dim)
    # cos, sin: (seq_len, head_dim) → unsqueeze to (seq_len, 1, head_dim)
    if cos.dim() == 2:
        cos = cos.unsqueeze(1)
        sin = sin.unsqueeze(1)
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed

def custom_forward(
    self,
    hidden_states: torch.Tensor,
    cu_seqlens: torch.Tensor,
    rotary_pos_emb: Optional[torch.Tensor] = None,
    position_embeddings: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
) -> torch.Tensor:
    seq_length = hidden_states.shape[0]
    q, k, v = self.qkv(hidden_states).reshape(
        seq_length, 3, self.num_heads, -1).permute(1, 0, 2, 3).unbind(0)

    if position_embeddings is None:
        emb = torch.cat((rotary_pos_emb, rotary_pos_emb), dim=-1)
        cos = emb.cos().float()
        sin = emb.sin().float()
    else:
        cos, sin = position_embeddings
        cos = cos.to(torch.float)
        sin = sin.to(torch.float)

    q, k = apply_rotary_pos_emb_custom(q.float(), k.float(), cos, sin)
    q = q.type_as(hidden_states)
    k = k.type_as(hidden_states)

    max_seqlen = (cu_seqlens[1:] - cu_seqlens[:-1]).max().item()
    attn_output = flash_attn_varlen_func(
        q, k, v, cu_seqlens, cu_seqlens, max_seqlen, max_seqlen
    ).reshape(seq_length, -1)
    attn_output = self.proj(attn_output)
    return attn_output

Qwen2_5_VLVisionFlashAttention2.forward = custom_forward
print("Flash attention patch applied.") 
import sys
sys.path.insert(0, "/root/iqa_unified")
from config import get_default_config
from model_utils import (load_model_and_processor, generate_response, parse_answer,
                          extract_scores, extract_degradation, parse_thinking,
                          clear_gpu_memory, print_gpu_memory)
from prompts import UNIFIED_ASSESSMENT_PROMPT, build_single_image_message
import torch, json
import numpy as np
from PIL import Image

# Verify GPU
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} — "
          f"{torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB")

config = get_default_config()
config.model.use_4bit = False   # A100 40GB — full bf16, no quantization needed

model, processor = load_model_and_processor(config.model)

test_image = Image.fromarray(np.random.randint(50, 200, (512, 512, 3), dtype=np.uint8))
messages   = build_single_image_message(test_image, UNIFIED_ASSESSMENT_PROMPT)
response   = generate_response(model, processor, messages, max_new_tokens=256, do_sample=False)
print("Sample response:", response[:200])
print("Parsed:", json.dumps(parse_answer(response), indent=2))
clear_gpu_memory()
print("Model verification complete!")

Flash attention patch applied.
PyTorch: 2.8.0+cu129
CUDA: True
  GPU 0: NVIDIA A100-SXM4-40GB — 42.4 GB


preprocessor_config.json:   0%|          | 0.00/575 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

score_degradation/model-00002-of-00004.s(…):   0%|          | 0.00/4.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/311 [00:00<?, ?B/s]

Could not find the bitsandbytes CUDA binary at PosixPath('/usr/local/lib/python3.12/site-packages/bitsandbytes/libbitsandbytes_cuda129.so')
The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.


trainable params: 23,794,688 || all params: 8,315,961,344 || trainable%: 0.2861
Model loaded!
  GPU 0: 16.63 GB allocated, 16.64 GB reserved
Sample response: <think>
The image provided seems to be an example of extreme digital distortion. It lacks discernible content, and the colors appear to be random noise. This kind of image is often used to demonstrate
Parsed: {
  "quality": 0.1,
  "authenticity": 0.1,
  "correspondence": 0.1,
  "distortion_class": "noise",
  "severity": "catastrophic"
}
Model verification complete!


In [15]:
import sys
sys.path.insert(0, "/root/iqa_unified")
import numpy as np
from iqa_datasets import create_combined_dataset, create_dataloaders

DATASET_CONFIGS = [
    {"name": "aigciqa2023", "dir": "/root/aigciqa2023",   "max_samples": 2000},
    {"name": "kadid10k",    "dir": "/root/kadid10k",      "max_samples": 2000},
]

combined = create_combined_dataset(DATASET_CONFIGS, base_dir="", seed=42)
print(f"Total samples: {len(combined)}")

train_loader, val_loader = create_dataloaders(
    combined, batch_size=1, num_workers=4, val_split=0.1, seed=42)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

  AIGCIQA2023 loaded with encoding=latin-1, rows=2400


Loading AIGCIQA2023: 100%|██████████████████████████████████████████████| 2000/2000 [00:00<00:00, 16531.52it/s]


  AIGCIQA2023: 2000 loaded, 0 skipped
  Loaded aigciqa2023: 2000 samples


Loading KADID10k: 100%|██████████████████████████████████████████████████| 2000/2000 [00:00<00:00, 6515.97it/s]

  KADID10k: 2000 loaded, 0 skipped
  Loaded kadid10k: 2000 samples
Total samples: 4000
Train batches: 3600, Val batches: 400


In [ ]:
import os, sys, json, importlib
os.environ["PYTHONUNBUFFERED"] = "1"
sys.path.insert(0, "/root/iqa_unified")

import grpo_trainer, rewards
importlib.reload(rewards)
importlib.reload(grpo_trainer)

from grpo_trainer import GRPOTrainer
from config import get_default_config
from peft import PeftModel

config = get_default_config()
config.model.use_4bit = False 
# All A100 settings already in config.py defaults — no overrides needed
config.grpo.num_epochs        = 1   
config.grpo.num_groups        = 4
# config.grpo.num_iterations    = 2   ✅ default
# config.grpo.gradient_accumulation_steps = 2  ✅ default

ckpt = "/checkpoints/ckpt_latest.pt"
config.resume_from = ckpt if os.path.exists(ckpt) else None

print("=" * 60)
print("GRPO Training Config:")
print(f"  Epochs:        {config.grpo.num_epochs}")
print(f"  Groups (G):    {config.grpo.num_groups}")
print(f"  Iterations:    {config.grpo.num_iterations}")
print(f"  Grad accum:    {config.grpo.gradient_accumulation_steps}")
print(f"  Learning rate: {config.grpo.learning_rate}")
print(f"  4-bit:         {config.model.use_4bit}")
print(f"  LoRA:          {config.model.use_lora}")
print(f"  Mixed prec:    {config.mixed_precision}")
print(f"  Attn impl:     {config.model.attn_implementation}")
print(f"  Resume from:   {config.resume_from or 'fresh start'}")
print("Pure GRPO — per-token loss, additive rewards, no KL")
print("=" * 60)

assert isinstance(model, PeftModel) == config.model.use_lora, \
    f"Model/config mismatch: PeftModel={isinstance(model, PeftModel)}, use_lora={config.model.use_lora}"
print(f"Model type check passed: PeftModel={isinstance(model, PeftModel)}")

trainer = GRPOTrainer(
    model=model, processor=processor, config=config,
    train_loader=train_loader, val_loader=val_loader,
)

train_results = trainer.train()

os.makedirs("/root/results", exist_ok=True)
log_path = "/root/results/training_log.json"
with open(log_path, "w") as f:
    json.dump(train_results, f, indent=2)
print(f"\nTraining log saved: {log_path}")
print(f"Best reward:  {train_results['best_reward']:.4f}")
print(f"Total time:   {train_results['total_time']/60:.1f} min")

GRPO Training Config:
  Epochs:        1
  Groups (G):    4
  Iterations:    2
  Grad accum:    2
  Learning rate: 1e-05
  4-bit:         False
  LoRA:          True
  Mixed prec:    bf16
  Attn impl:     flash_attention_2
  Resume from:   fresh start
Pure GRPO — per-token loss, additive rewards, no KL
Model type check passed: PeftModel=True
Trainable parameters: 23,794,688
GRPO TRAINING  |  bf16  |  lr=1e-05
Epochs: 1  Groups: 4  Iterations: 2  4-bit: False  LoRA: True
GRPO — wide temperature diversity, smooth rewards, no entropy term


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

  [DEBUG step 0] rewards: ['0.3379', '0.3544', '0.2109', '0.3756']
  [DEBUG step 0] advantages: ['0.2452', '0.4673', '-1.4652', '0.7527']
  [DEBUG step 0] reward_std: 0.0742  skipped=no
[  0.0%] Ep0    1/3600 | R=0.320 L=-0.010536 | GPU0=16.7GB GPU1=0.0GB | step=61.8s ETA=14:41:25
  [DEBUG step 1] rewards: ['0.4560', '0.4562', '0.4560', '0.4560']
  [DEBUG step 1] advantages: ['0.0000', '0.0000', '0.0000', '0.0000']
  [DEBUG step 1] reward_std: 0.0001  skipped=yes
[  0.1%] Ep0    2/3600 | R=0.388 L=-0.005268 | GPU0=16.7GB GPU1=0.0GB | step=63.7s ETA=15:08:56
  [DEBUG step 2] rewards: ['0.3279', '0.4279', '0.3229', '0.3279']
  [DEBUG step 2] advantages: ['-0.4657', '1.4954', '-0.5641', '-0.4657']
  [DEBUG step 2] reward_std: 0.0509  skipped=no
[  0.1%] Ep0    3/3600 | R=0.376 L=-0.002588 | GPU0=16.8GB GPU1=0.0GB | step=48.7s ETA=10:19:17
[  0.1%] Ep0    4/3600 | R=0.397 L=-0.003234 | GPU0=16.8GB GPU1=0.0GB | step=79.6s ETA=15:36:27
[  0.1%] Ep0    5/3600 | R=0.446 L=-0.004391 | GPU0=16.8

In [ ]:
import shutil, os

# List checkpoints
print("Checkpoints saved:")
for f in sorted(os.listdir("/checkpoints")):
    size = os.path.getsize(f"/checkpoints/{f}") / 1e6
    print(f"  {f}: {size:.1f} MB")

# Create zip
shutil.make_archive("/root/checkpoints_ep0", "zip", "/checkpoints")
size = os.path.getsize("/root/checkpoints_ep0.zip") / 1e6
print(f"\ncheckpoints_ep0.zip: {size:.1f} MB")

In [ ]:
import torch, sys, importlib
sys.path.insert(0, "/root/iqa_unified")

# Load best checkpoint
payload = torch.load("/checkpoints/ckpt_best.pt", map_location="cuda")
model.load_state_dict(payload["lora_state_dict"], strict=False)
print(f"Loaded best checkpoint — epoch: {payload['epoch']}, step: {payload['step']}, "
      f"best_reward: {payload['best_reward']:.4f}")

# Reload modules to pick up any changes
import evaluate, metrics
importlib.reload(metrics)
importlib.reload(evaluate)
from evaluate import IQAEvaluator
from config import get_default_config

config = get_default_config()
config.model.use_4bit = False

evaluator = IQAEvaluator(model, processor, config)

EVAL_DATASET_CONFIGS = [
    {"name": "aigciqa2023", "dir": "/root/aigciqa2023"},
    {"name": "kadid10k",    "dir": "/root/kadid10k"},
]

eval_results = evaluator.run_full_evaluation(
    EVAL_DATASET_CONFIGS, max_samples=100, run_adversarial=False
)

print("\n" + "=" * 60)
print("FINAL RESULTS SUMMARY")
print("=" * 60)
for ds, res in eval_results["on_manifold"].items():
    if "error" in res:
        print(f"\n{ds}: ERROR — {res['error']}")